Here we illustrate with a toy example the importance of balanced training

In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
import torch.nn.functional as F
from training.models.losses import ArcFaceLoss
from training.models.arcface import MetricLearningModel
import seaborn as sns
import pandas as pd
from tqdm import tqdm
from evaluation.visualize import (
    plot_rejection_scores,
)
import gpytorch
from torch.utils.data import DataLoader
from utils_notebooks import predict_features, compute_distance_and_visualize
from botorch.models import SingleTaskGP
from botorch.models.transforms import Normalize, Standardize
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.optim import optimize_acqf
from botorch.acquisition import LogExpectedImprovement

%load_ext autoreload
%autoreload 2

### 1. Train arcface on mnist

In [2]:
from toy_scf.models import (
    Backbone,
    train_arcface,
    compute_cosine_sim,
    predict_accuracy,
    load_arcface_model,
    load_scf_model,
    train_scf,
    compute_kappa,
)

In [3]:
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
)
mnist_ds_train = datasets.MNIST(
    root="/app/datasets/mnist", train=True, download=True, transform=transform
)
mnist_ds_test = datasets.MNIST(
    root="/app/datasets/mnist", train=False, download=True, transform=transform
)

#### 1.1 Train Arcface on full ds

In [ ]:
NUM_FEATURES = 2

backbone_model = Backbone(num_features=NUM_FEATURES)
arcface_loss = ArcFaceLoss()

arcface_model = MetricLearningModel(
    backbone_model,
    arcface_loss,
    num_labels=10,
    train_set=mnist_ds_train,
    val_set=mnist_ds_test,
    batch_size=400,
    num_workers=4,
    num_features=NUM_FEATURES,
)
exp_name = "arcface_full_mnist"
# train_arcface(arcface_model, f"/app/outputs/{exp_name}", exp_name, max_epoch=100)

##### Cosine similarity distribution

In [ ]:
train_dl = DataLoader(
    mnist_ds_train,
    batch_size=128,
    shuffle=False,
    drop_last=False,
    num_workers=32,
)
test_dl = DataLoader(
    mnist_ds_test,
    batch_size=128,
    shuffle=False,
    drop_last=False,
    num_workers=32,
)
arcface_model = load_arcface_model(
    mnist_ds_train,
    mnist_ds_test,
    "/app/outputs/arcface_full_mnist/ckpt/epoch=89-val_loss=1.24.ckpt",
    visualize=True,
)
file_name = "full_mnist"
compute_cosine_sim(arcface_model, train_dl, test_dl, file_name)
cosine_sim_train = np.load(f"outputs/train_{file_name}_cosine_sim.npy")
cosine_sim_test = np.load(f"outputs/test_{file_name}_cosine_sim.npy")
data = {
    "test": (np.arccos(cosine_sim_test) / (2 * np.pi)) * 360,
    "train": (np.arccos(cosine_sim_train) / (2 * np.pi)) * 360,
}
sns.displot(
    data,
    kind="kde",
    log_scale=False,
    common_norm=False,
)

#### 1.2 Train Arcface on filtered ds  

In [ ]:
np.sum((np.arccos(cosine_sim_train) / (2 * np.pi)) * 360 > 18), len(cosine_sim_train)

In [ ]:
EPS = 1e-6
angle = 18
angles = (np.arccos(np.clip(cosine_sim_train, -1.0 + EPS, 1.0 - EPS)) / (np.pi)) * 180
mnist_ds_train_filtered = torch.utils.data.Subset(
    mnist_ds_train,
    np.where(angles < angle)[0],
)
len(mnist_ds_train_filtered)

In [ ]:
NUM_FEATURES = 2

backbone_model = Backbone(num_features=NUM_FEATURES)
arcface_loss = ArcFaceLoss()

arcface_model = MetricLearningModel(
    backbone_model,
    arcface_loss,
    num_labels=10,
    train_set=mnist_ds_train_filtered,
    val_set=mnist_ds_test,
    batch_size=400,
    num_workers=4,
    num_features=NUM_FEATURES,
)
exp_name = f"arcface_angle_{angle}_filtered_mnist"
# train_arcface(arcface_model, f"/app/outputs/{exp_name}", exp_name, max_epoch=100)

In [ ]:
train_dl = DataLoader(
    mnist_ds_train_filtered,
    batch_size=128,
    shuffle=False,
    drop_last=False,
    num_workers=32,
)
test_dl = DataLoader(
    mnist_ds_test,
    batch_size=128,
    shuffle=False,
    drop_last=False,
    num_workers=32,
)
arcface_model = load_arcface_model(
    mnist_ds_train_filtered,
    mnist_ds_test,
    "/app/outputs/arcface_angle_18_filtered_mnist/ckpt/epoch=99-val_loss=1.09.ckpt",
)
file_name = f"angle_{angle}_filtered_mnist"
compute_cosine_sim(arcface_model, train_dl, test_dl, file_name)
cosine_sim_train = np.load(f"outputs/train_{file_name}_cosine_sim.npy")
cosine_sim_test = np.load(f"outputs/test_{file_name}_cosine_sim.npy")
data = {
    "test": (np.arccos(cosine_sim_test) / (2 * np.pi)) * 360,
    "train": (np.arccos(cosine_sim_train) / (2 * np.pi)) * 360,
}
sns.displot(
    data,
    kind="kde",
    log_scale=False,
    common_norm=False,
)

#### 1.3 Compute error probabilities on train MNIST and clean MNIST
Here we use arcface model trained on clean MNIST

In [ ]:
arcface_model = load_arcface_model(
    mnist_ds_train,
    mnist_ds_test,
    "/app/outputs/arcface_angle_18_filtered_mnist/ckpt/epoch=99-val_loss=1.09.ckpt",
)
arcface_model.eval()

train_dl = DataLoader(
    mnist_ds_train,
    batch_size=128,
    shuffle=False,
    drop_last=False,
    num_workers=32,
)
train_dl_filtered = DataLoader(
    mnist_ds_train_filtered,
    batch_size=128,
    shuffle=False,
    drop_last=False,
    num_workers=32,
)

predicted_train_features, train_labels = predict_features(arcface_model, train_dl)
predicted_train_features_filtered, train_labels_filtered = predict_features(
    arcface_model, train_dl_filtered
)
softmax_weights = (
    F.normalize(arcface_model.softmax_weights, dim=1).detach().cpu().numpy()
)

In [ ]:
arcface_model

In [ ]:
cosine_sim = predicted_train_features @ softmax_weights.T
s = 5
p_z = np.exp(s * cosine_sim)
p_z_filtered = np.exp(s * predicted_train_features_filtered @ softmax_weights.T)
true_class_prob = p_z[np.arange(p_z.shape[0]), train_labels] / np.sum(p_z, axis=1)
true_class_prob_filtered = p_z_filtered[
    np.arange(p_z_filtered.shape[0]), train_labels_filtered
] / np.sum(p_z_filtered, axis=1)

data = {
    "error probability": 1 - true_class_prob,
    "error probability filtered": 1 - true_class_prob_filtered,
}
sns.displot(
    data,
    kind="kde",
    log_scale=False,
    common_norm=False,
)

In [ ]:
np.mean(1 - true_class_prob), np.mean(1 - true_class_prob_filtered)

In [ ]:
np.median(1 - true_class_prob), np.median(1 - true_class_prob_filtered)

In [ ]:
cosine_sim = (predicted_train_features @ softmax_weights.T)[
    np.arange(predicted_train_features.shape[0]), train_labels
]
cosine_sim_filtered = (predicted_train_features_filtered @ softmax_weights.T)[
    np.arange(predicted_train_features_filtered.shape[0]), train_labels_filtered
]
data = {
    "cosine sim": cosine_sim,
    "cosine sim filtered": cosine_sim_filtered,
}
sns.displot(
    data,
    kind="kde",
    log_scale=False,
    common_norm=False,
)

#### 2.1 Train SCF on full mnist

In [16]:
# arcface_model = load_arcface_model(
#     mnist_ds_train,
#     mnist_ds_test,
#     "/app/outputs/arcface_angle_18_filtered_mnist/ckpt/epoch=99-val_loss=1.09.ckpt",
# )
# arcface_model.eval()
# torch.save(arcface_model.softmax_weights.detach().cpu(), "outputs/softmax_weights.pt")

In [17]:
# train_scf('scf_full_mnist', mnist_ds_train, mnist_ds_test, '/app/outputs/arcface_angle_18_filtered_mnist/ckpt/epoch=99-val_loss=1.09.ckpt')

#### 2.2 Train SCF on filtered dataset

In [18]:
exp_name = f"scf_angle_{angle}_filtered_mnist"
# train_scf(
#     exp_name,
#     mnist_ds_train_filtered,
#     mnist_ds_test,
#     "/app/outputs/arcface_angle_18_filtered_mnist/ckpt/epoch=99-val_loss=1.09.ckpt",
# )

#### 2.3 Train SCF on DirtyMNIST

In [19]:
# import ddu_dirty_mnist

# # transform_dirty = transforms.Compose(
# #     [transforms.Normalize((0.1307,), (0.3081,))]
# # )
# dirty_mnist_train = ddu_dirty_mnist.DirtyMNIST(
#     root="/app/datasets/dirty_mnist", train=True, download=True
# )
# dirty_mnist_test = ddu_dirty_mnist.DirtyMNIST(
#     root="/app/datasets/dirty_mnist", train=False, download=True
# )
# len(dirty_mnist_train), len(dirty_mnist_test)

# exp_name = f"scf_dirty_mnist"
# train_scf(
#     exp_name,
#     dirty_mnist_train,
#     mnist_ds_test,
#     "/app/outputs/arcface_angle_18_filtered_mnist/ckpt/epoch=99-val_loss=1.09.ckpt",
# )

#### 3.1 Compare UE methods

In [ ]:
arcface_path = (
    "/app/outputs/arcface_angle_18_filtered_mnist/ckpt/epoch=99-val_loss=1.09.ckpt"
)
scf_full = load_scf_model(
    mnist_ds_train,
    mnist_ds_test,
    scf_path="/app/outputs/scf_full_mnist/ckpt/2024-08-30 15:28:19.199679/epoch=99-step=15000.ckpt",
    arcface_path=arcface_path,
)

In [ ]:
(2700 / 60000) * 100

In [ ]:
full_features, full_labels, full_kappa = compute_kappa(scf_full, mnist_ds_test)

In [ ]:
scf_filtered = load_scf_model(
    mnist_ds_train,
    mnist_ds_test,
    scf_path="/app/outputs/scf_angle_18_filtered_mnist/ckpt/2024-08-30 15:36:36.239175/epoch=99-step=14400.ckpt",
    arcface_path=arcface_path,
)

In [ ]:
filtered_features, filtered_labels, filtered_kappa = compute_kappa(
    scf_filtered, mnist_ds_test
)

In [26]:
# scf_dirty_mnist = load_scf_model(
#     dirty_mnist_train,
#     mnist_ds_test,
#     scf_path="/app/outputs/scf_dirty_mnist/ckpt/2024-09-16 11:36:51.218096/epoch=99-step=30000.ckpt",
#     arcface_path=arcface_path,
# )
# dirty_mnist_features, dirty_mnist_labels, dirty_mnist_kappa = compute_kappa(
#     scf_dirty_mnist, mnist_ds_test
# )

In [ ]:
arcface_model = load_arcface_model(
    mnist_ds_train,
    mnist_ds_test,
    arcface_path,
)
softmax_weights = (
    torch.nn.functional.normalize(arcface_model.softmax_weights, dim=1)
    .detach()
    .cpu()
    .numpy()
)

#### TAR@FAR Curve

In [28]:
# sns.displot(
#     {"full": full_kappa, "filtered": filtered_kappa, "dirty mnist": dirty_mnist_kappa},
#     kind="kde",
#     log_scale=False,
#     common_norm=False,
# )

#### Train scf on augmented data

In [ ]:
from torchvision.transforms.v2 import GaussianNoise, GaussianBlur
from torchvision.transforms import v2

mnist_ds_train_vis = datasets.MNIST(
    root="/app/datasets/mnist", train=True, download=True
)
transform = v2.Compose([v2.ToTensor(), v2.Normalize((0.1307,), (0.3081,))])
transform_nose = v2.Compose(
    [
        v2.ToTensor(),
        v2.Normalize((0.1307,), (0.3081,)),
        GaussianNoise(mean=0.6, sigma=0.4),
    ]
)
transform_blur = v2.Compose(
    [
        v2.ToTensor(),
        v2.Normalize((0.1307,), (0.3081,)),
        GaussianBlur(kernel_size=5, sigma=5),
    ]
)
transform_nose_blur = v2.Compose(
    [
        v2.ToTensor(),
        v2.Normalize((0.1307,), (0.3081,)),
        v2.RandomApply(
            [
                GaussianNoise(mean=0.3, sigma=0.5),
                GaussianBlur(kernel_size=5, sigma=(1, 5)),
            ],
            p=0.5,
        ),
    ]
)
image_id = 0
image = transform(mnist_ds_train_vis[image_id][0])[0, ...]
image_nose = transform_nose(mnist_ds_train_vis[image_id][0])[0, ...]
image_blur = transform_blur(mnist_ds_train_vis[image_id][0])[0, ...]
image_nose_blur = transform_nose_blur(mnist_ds_train_vis[image_id][0])[0, ...]

fig, axes = plt.subplots(1, 4)  # figsize=(1.5 * num_col, 2 * num_row)#
axes[0].imshow(image)
axes[1].imshow(image_nose)
axes[2].imshow(image_blur)
axes[3].imshow(image_nose_blur)

In [ ]:
def get_aug_dataset(aug_probability, sigma, kernel_size):
    transform_nose_blur = v2.Compose(
        [
            v2.ToTensor(),
            v2.Normalize((0.1307,), (0.3081,)),
            v2.RandomApply(
                [
                    # GaussianNoise(mean=0.3, sigma=0.5),
                    GaussianBlur(kernel_size=kernel_size, sigma=sigma),
                ],
                p=aug_probability,
            ),
        ]
    )

    cosine_sim_train = np.load(f"outputs/train_full_mnist_cosine_sim.npy")
    mnist_ds_train_aug = datasets.MNIST(
        root="/app/datasets/mnist",
        train=True,
        download=True,
        transform=transform_nose_blur,
    )
    EPS = 1e-6
    angle = 18
    angles = (
        np.arccos(np.clip(cosine_sim_train, -1.0 + EPS, 1.0 - EPS)) / (np.pi)
    ) * 180
    mnist_ds_train_filtered_aug = torch.utils.data.Subset(
        mnist_ds_train_aug,
        np.where(angles < angle)[0],
    )
    return mnist_ds_train_filtered_aug


mnist_ds_train_filtered_aug = get_aug_dataset(
    aug_probability=0.5, sigma=4, kernel_size=5
)
len(mnist_ds_train_filtered_aug)

In [31]:
def compute_mean_accuracy(aug_probability, sigma, kernel_size):
    mnist_ds_train_filtered_aug = get_aug_dataset(
        aug_probability=aug_probability, sigma=sigma, kernel_size=kernel_size
    )
    exp_name = f"scf_angle_{angle}_filtered_aug_blur_p={np.round(aug_probability, 2)}-sig={np.round(sigma,2)}_kernel={np.round(kernel_size, 2)}"
    scf_aug = train_scf(
        exp_name,
        mnist_ds_train_filtered_aug,
        mnist_ds_test,
        "/app/outputs/arcface_angle_18_filtered_mnist/ckpt/epoch=99-val_loss=1.09.ckpt",
    )
    scf_aug.eval()
    aug_features, aug_labels, aug_kappa = compute_kappa(scf_aug, mnist_ds_test)
    fractions = [0, 0.3, 10]
    fractions_linspace = np.linspace(fractions[0], fractions[1], fractions[2])
    accuracies = []
    unc_indexes = np.argsort(-aug_kappa)
    for fraction in fractions_linspace:
        good_idx = unc_indexes[: int((1 - fraction) * aug_kappa.shape[0])]
        accuracies.append(
            predict_accuracy(
                features=aug_features[good_idx],
                labels=aug_labels[good_idx],
                softmax_weights=softmax_weights,
            )
        )
    return np.mean(accuracies) * fractions[-2]


# accuracy = compute_mean_accuracy(aug_probability=0.5, sigma=4, kernel_size=5)

In [32]:
# from botorch.models import SingleTaskGP
# from botorch.models.transforms import Normalize, Standardize
# from botorch.fit import fit_gpytorch_mll
# from gpytorch.mlls import ExactMarginalLogLikelihood
# from botorch.optim import optimize_acqf
# from botorch.acquisition import LogExpectedImprovement

# train_X = torch.tensor([[0.5, 4]], dtype=torch.double)
# Y = torch.tensor([[0.283645]], dtype=torch.double)
# num_steps = 10
# for i in range(num_steps):
#     gp = SingleTaskGP(
#         train_X=train_X,
#         train_Y=Y,
#         input_transform=Normalize(d=2),
#         outcome_transform=Standardize(m=1),
#     )
#     mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
#     fit_gpytorch_mll(mll)
#     logEI = LogExpectedImprovement(model=gp, best_f=Y.max())
#     bounds = torch.tensor([[0, 0.1], [1, 10]]).to(torch.double)
#     candidate, acq_value = optimize_acqf(
#         logEI,
#         bounds=bounds,
#         q=1,
#         num_restarts=5,
#         raw_samples=20,
#     )

#     accuracy = compute_mean_accuracy(
#         aug_probability=candidate[0, 0].item(),
#         sigma=candidate[0, 1].item(),
#         kernel_size=5,
#     )
#     train_X = torch.concatenate([train_X, candidate])
#     Y = torch.concatenate([Y, torch.tensor([[accuracy]], dtype=torch.double)])
#     print(f"iteration {i}: accuracy {np.round(accuracy, 4)}")

In [33]:
# train_X.shape, Y.shape
# torch.cat([train_X, Y], dim=-1)

In [66]:
data_vis = np.array(
    [
        [0.5000, 4.0000, 0.2836],
        [0.7531, 8.1725, 0.2831],
        [0.4277, 2.9401, 0.2838],
        [0.4394, 1.3498, 0.2838],
        [0.3426, 2.0090, 0.2834],
        [0.0000, 10.0000, 0.2829],
        [0.4803, 1.7045, 0.2833],
        [0.3379, 5.1669, 0.2836],
        [0.8109, 0.8779, 0.2839],
        [0.9960, 0.1000, 0.2825],
        [0.6756, 1.7220, 0.2831],
    ]
)

train_X_vis = torch.tensor(data_vis[:, [0, 1]], dtype=torch.double)
Y_vis = torch.tensor(data_vis[:, [2]], dtype=torch.double)

In [ ]:
Y_vis.shape

In [ ]:
gp = SingleTaskGP(
    train_X=train_X_vis,
    train_Y=Y_vis,
    input_transform=Normalize(d=2),
    outcome_transform=Standardize(m=1),
)
mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
fit_gpytorch_mll(mll)

In [69]:
# Get into evaluation (predictive posterior) mode
gp.eval()
gp.likelihood.eval()

probabilities = torch.linspace(0, 1, 30)
sigmas = torch.linspace(0, 10, 30)
grid_x, grid_y = torch.meshgrid(probabilities, sigmas, indexing="ij")
product = torch.cartesian_prod(probabilities, sigmas)
# Make predictions by feeding model through likelihood
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    observed_pred = gp.posterior(product)
lower, upper = observed_pred.confidence_region()

In [93]:
logEI = LogExpectedImprovement(model=gp, best_f=Y_vis.max())
acq = logEI(product[:, None, :])

In [ ]:
acq

In [70]:
# Get upper and lower confidence bounds

# ax.plot(test_x.numpy(), , "b")
# # Shade between the lower and upper confidence bounds
# ax.fill_between(test_x.numpy(), lower.numpy(), upper.numpy(), alpha=0.5)

In [ ]:
observed_pred.mean.numpy()

In [ ]:
observed_pred.mean.numpy().shape

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(1)


fig = go.Figure()
trace1 = go.Mesh3d(
    x=(product[:, 0]),
    y=(product[:, 1]),
    z=(observed_pred.mean.numpy()[:, 0]),
    opacity=0.5,
    color="rgba(244,22,100,0.6)",
)
fig.add_trace(trace1)

trace_lower = go.Mesh3d(
    x=(product[:, 0]),
    y=(product[:, 1]),
    z=(lower.numpy()),
    opacity=0.5,
    color="blue",
)
fig.add_trace(trace_lower)

trace_upper = go.Mesh3d(
    x=(product[:, 0]),
    y=(product[:, 1]),
    z=(upper.numpy()),
    opacity=0.5,
    color="blue",
)
fig.add_trace(trace_upper)

trace2 = go.Scatter3d(
    x=train_X_vis[:, 0], y=train_X_vis[:, 1], z=Y_vis[:, 0], mode="markers"
)

fig.add_trace(trace2)


fig.update_layout(
    scene=dict(xaxis_title="p", yaxis_title="sigma", zaxis_title="Accuracy"),
    # zaxis=dict(title="accuracy"),
)
fig.show()

In [ ]:
fig = go.Figure()
trace_acq = go.Mesh3d(
    x=(product[:, 0]),
    y=(product[:, 1]),
    z=(acq.detach().numpy()),
    opacity=0.5,
    color="blue",
)
fig.add_trace(trace_acq)

fig.update_layout(
    scene=dict(xaxis_title="p", yaxis_title="sigma", zaxis_title="Acq"),
    # zaxis=dict(title="accuracy"),
)
fig.show()

In [ ]:
# scf_aug = load_scf_model(
#     mnist_ds_train,
#     mnist_ds_test,
#     scf_path="/app/outputs/scf_angle_18_filtered_aug_mnist/ckpt/2024-09-02 11:02:15.881051/epoch=99-step=14400.ckpt",
#     arcface_path=arcface_path,
# )
scf_aug = load_scf_model(
    mnist_ds_train,
    mnist_ds_test,
    scf_path="/app/outputs/scf_angle_18_filtered_aug_blur_mnist/ckpt/2024-10-08 14:01:37.482510/epoch=19-step=2880.ckpt",
    arcface_path=arcface_path,
)
aug_features, aug_labels, aug_kappa = compute_kappa(scf_aug, mnist_ds_test)

In [ ]:
scf_aug_best = load_scf_model(
    mnist_ds_train,
    mnist_ds_test,
    scf_path="/app/outputs/scf_mnist_aug_search/scf_angle_18_filtered_aug_blur_p=0.81-sig=0.88_kernel=5/ckpt/2024-10-08 15:15:25.511457/epoch=19-step=2880.ckpt",
    arcface_path=arcface_path,
)
aug_features_best, aug_labels_best, aug_kappa_best = compute_kappa(
    scf_aug_best, mnist_ds_test
)

In [ ]:
from toy_scf.models import get_rejection_accuracy

fractions = [0, 0.3, 10]
fractions_linspace = np.linspace(fractions[0], fractions[1], fractions[2])


model_names = [
    "full_mnist_scf",
    "filtered_mnist_scf",
    "filtered_mnist_aug_scf",
    "filtered_mnist_aug_scf_best",
]  # , "dirty_mnist"]
features_list = [
    (full_features, full_labels, full_kappa),
    (filtered_features, filtered_labels, filtered_kappa),
    (aug_features, aug_labels, aug_kappa),
    (aug_features_best, aug_labels_best, aug_kappa_best),
]
fig = get_rejection_accuracy(model_names, features_list, fractions, softmax_weights)
fig.show()

In [ ]:
np.mean(metric_scores[-1][1]) * 0.3

In [ ]:
np.mean(metric_scores[-2][1]) * 0.3

In [ ]:
metric_scores

### Augmentation papameters search

In [ ]:
# predicted_test_features, test_labels = predict_features(arcface_model, test_dl)
pairs = pd.read_csv("outputs/pairs.csv")


def distance_function(X_1, X_2):
    return np.sum(X_1 * X_2, axis=1)


p1 = pairs["first_template"].values
p2 = pairs["second_template"].values

batch_size = 10000
steps = int(np.ceil(len(p1) / batch_size))
scores = []
unc_full = []
unc_filtered = []
unc_aug = []
for id in tqdm(range(steps), "Verification"):
    feat1 = full_features[p1[id * batch_size : (id + 1) * batch_size]]
    feat2 = full_features[p2[id * batch_size : (id + 1) * batch_size]]
    unc1 = full_kappa[p1[id * batch_size : (id + 1) * batch_size]]
    unc2 = full_kappa[p2[id * batch_size : (id + 1) * batch_size]]
    unc3 = filtered_kappa[p1[id * batch_size : (id + 1) * batch_size]]
    unc4 = filtered_kappa[p2[id * batch_size : (id + 1) * batch_size]]
    unc5 = aug_kappa[p1[id * batch_size : (id + 1) * batch_size]]
    unc6 = aug_kappa[p2[id * batch_size : (id + 1) * batch_size]]
    unc_full.extend(-np.minimum(unc1, unc2))
    unc_filtered.extend(-np.minimum(unc3, unc4))
    unc_aug.extend(-np.minimum(unc5, unc6))
    scores.extend(distance_function(feat1, feat2))
scores = np.array(scores)
unc_full = np.array(unc_full)
unc_filtered = np.array(unc_filtered)
unc_aug = np.array(unc_aug)

In [36]:
# from utils_notebooks import tar_far_curve
# tar_far_curve(scores, pairs)

In [ ]:
from evaluation.uncertainty_metrics import DisposeBasedOnUncVerif
from evaluation.metrics import TarFar

fractions = [0, 0.3, 5]
metric_to_monitor = TarFar(far_range=[-6, 0, 0.1], display_fars=[1e-3, 1e-2])
unc_metric = DisposeBasedOnUncVerif(
    fractions=fractions, metric_to_monitor=metric_to_monitor
)
full_unc_metrics = unc_metric(
    scores=scores, labels=pairs["is_positive"].values, predicted_unc=unc_full
)
filtered_unc_metrics = unc_metric(
    scores=scores, labels=pairs["is_positive"].values, predicted_unc=unc_filtered
)
aug_unc_metrics = unc_metric(
    scores=scores, labels=pairs["is_positive"].values, predicted_unc=unc_aug
)

for metric_name in full_unc_metrics:
    if metric_name == "fractions":
        continue

    metric_scores = []
    model_names = []
    model_names.append("scf_full")
    model_names.append("scf_filtered")
    model_names.append("filtered_aug")
    metric_pretty_name = metric_name.split(":")[-1]
    metric_scores.append((full_unc_metrics["fractions"], full_unc_metrics[metric_name]))
    metric_scores.append(
        (full_unc_metrics["fractions"], filtered_unc_metrics[metric_name])
    )
    metric_scores.append((full_unc_metrics["fractions"], aug_unc_metrics[metric_name]))
    fig, rejection_metric_values = plot_rejection_scores(
        scores=metric_scores,
        names=model_names,
        y_label=f"{metric_pretty_name}",
    )
    fig.show()
    # plt.close(fig)

### Augmentation papameters search

In [ ]:
# mnist_ds_train_vis = datasets.MNIST(
#     root="/app/datasets/mnist", train=True, download=True
# )
# num_row = 5
# num_col = 5

# # plot images
# fig, axes = plt.subplots(num_row, num_col, figsize=(1.5 * num_col, 2 * num_row))
# for i in range(num_row * num_col):
#     id = conf_sort[i]
#     ax = axes[i // num_col, i % num_col]
#     ax.imshow(mnist_ds_train_vis[id][0], cmap="gray")
#     ax.set_title(f"Conf: {str(np.round(train_kappa[id], 4))}")
# plt.tight_layout()
# plt.show()

In [ ]:
# import matplotlib.pyplot as plt

# # mnist_ds_train_vis = datasets.MNIST(
# #     root="/app/datasets/mnist", train=True, download=True)
# num_row = 10
# num_col = 5

# # plot images
# fig, axes = plt.subplots(num_row, num_col, figsize=(1.5 * num_col, 2 * num_row))
# for i in range(num_row * num_col):
#     id = conf_sort[-i - 1]
#     ax = axes[i // num_col, i % num_col]
#     ax.imshow(mnist_ds_train_vis[id][0], cmap="gray")
#     ax.set_title(f"Conf: {str(np.round(train_kappa[id], 4))}")
# plt.tight_layout()
# plt.show()

#### Verification protocol

In [ ]:
# test_dl = DataLoader(
#     mnist_ds_test,
#     batch_size=128,
#     shuffle=False,
#     drop_last=False,
#     num_workers=32,
# )
# predicted_test_features, test_labels = predict_features(arcface_model, test_dl)
# first_template = []
# second_template = []
# is_positive = []
# for i in tqdm(range(len(mnist_ds_test))):
#     for j in range(len(mnist_ds_test)):
#         if i <= j: continue
#         first_template.append(i)
#         second_template.append(j)
#         is_positive.append(test_labels[i] == test_labels[j])
# first_template = np.array(first_template)
# second_template = np.array(second_template)
# is_positive = np.array(is_positive)
# pairs = pd.DataFrame(
#     {
#         "first_template": first_template,
#         "second_template": second_template,
#         "is_positive": is_positive,
#     }
# )
# pairs.to_csv("pairs.csv", index=False)

# predicted_test_features, test_labels = predict_features(arcface_model, test_dl)
# pairs = pd.read_csv("outputs/pairs.csv")
# def distance_function(X_1, X_2):
#     return np.sum(X_1 * X_2, axis=1)


# p1 = pairs["first_template"].values
# p2 = pairs["second_template"].values

# batch_size = 10000
# steps = int(np.ceil(len(p1) / batch_size))
# scores = []
# for id in tqdm(range(steps), "Verification"):
#     feat1 = predicted_test_features[p1[id * batch_size : (id + 1) * batch_size]]
#     feat2 = predicted_test_features[p2[id * batch_size : (id + 1) * batch_size]]
#     scores.extend(distance_function(feat1, feat2))
# scores = np.array(scores)

# from evaluation.visualize import draw_score_distr_plot

# is_positive = pairs["is_positive"].values

# true_match_scores = scores[is_positive]
# wrong_match_scores = scores[is_positive == 0]
# scores_distr = {
#     "Истинная пара": true_match_scores,
#     "Ложная пара": wrong_match_scores,
# }
# draw_score_distr_plot(
#     scores_distr=scores_distr,
#     positive_pair_name="Истинная пара",
#     negative_pair_name="Ложная пара",
# )